In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# 3GromvzkG1Pm7vXqKTzmetro5XC_2s6Xn7Nx3Wvffx8mFpmKk

In [3]:
pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
def generate_text(prompt, max_length=300):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [6]:
NGROK_TOKEN = "3GromvzkG1Pm7vXqKTzmetro5XC_2s6Xn7Nx3Wvffx8mFpmKk"   # use your ngrok token
API_KEY = "secret123"

In [7]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    return {
        "response": generate_text(
            data.get("prompt", ""),
            data.get("max_length", 300)
        )
    }

In [8]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://tidal-easily-diligence.ngrok-free.dev


INFO:     Started server process [322]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:33039 (Press CTRL+C to quit)
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.156.142.192:0 - "POST /generate HTTP/1.1"